In [9]:
from fedbiomed.common.training_plans import TorchTrainingPlan
from fedbiomed.common.datamanager import DataManager
from fedbiomed.common.dataset import MedicalFolderDataset
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage
 
import torch
import torch.nn as nn
from torch.optim import AdamW
from unet import UNet

In [10]:
# ############################################################################
#  Training plan (evaluation only)
##############################################################################

class UNetValidationPlan(TorchTrainingPlan):
 
    def init_model(self, model_args):
        return self.Net(model_args)
 
    def init_optimizer(self, optimizer_args):
        # lr = 0.0 -> the optimizer is a no-op. AdamW uses decoupled weight
        # decay (p -= lr * wd * p), so lr=0 also disables weight decay.
        return AdamW(self.model().parameters(),
                     lr=optimizer_args.get('lr', 0.0),
                     weight_decay=0.0)
 
    def init_dependencies(self):
        return [
            "from monai.transforms import (Compose, NormalizeIntensity, "
            "EnsureChannelFirst, Resize, AsDiscrete)",
            "import torch",
            "import torch.nn as nn",
            "import torch.nn.functional as F",
            "from fedbiomed.common.dataset import MedicalFolderDataset",
            "import numpy as np",
            "from torch.optim import AdamW",
            "from unet import UNet",
        ]
 
    class Net(nn.Module):
        def __init__(self, model_args: dict = {}):
            super().__init__()
            self.CHANNELS_DIMENSION = 1
            self.unet = UNet(
                in_channels=model_args.get('in_channels', 1),
                out_classes=model_args.get('out_classes', 2),
                dimensions=model_args.get('dimensions', 2),
                num_encoding_blocks=model_args.get('num_encoding_blocks', 5),
                out_channels_first_layer=model_args.get('out_channels_first_layer', 64),
                normalization=model_args.get('normalization', None),
                pooling_type=model_args.get('pooling_type', 'max'),
                upsampling_type=model_args.get('upsampling_type', 'conv'),
                preactivation=model_args.get('preactivation', False),
                residual=model_args.get('residual', False),
                padding=model_args.get('padding', 0),
                padding_mode=model_args.get('padding_mode', 'zeros'),
                activation=model_args.get('activation', 'ReLU'),
                initial_dilation=model_args.get('initial_dilation', None),
                dropout=model_args.get('dropout', 0),
                monte_carlo_dropout=model_args.get('monte_carlo_dropout', 0),
            )
 
        def forward(self, x):
            x = self.unet.forward(x)
            return F.softmax(x, dim=self.CHANNELS_DIMENSION)
 
    # ----------------------------- metrics --------------------------------- #
    @staticmethod
    def get_dice_score(output, target, epsilon=1e-9):
        """Per-sample, per-class Dice score. Returns a (batch, n_classes) tensor."""
        SPATIAL_DIMENSIONS = 2, 3, 4
        p0, g0 = output, target
        p1, g1 = 1 - p0, 1 - g0
        tp = (p0 * g0).sum(dim=SPATIAL_DIMENSIONS)
        fp = (p0 * g1).sum(dim=SPATIAL_DIMENSIONS)
        fn = (p1 * g0).sum(dim=SPATIAL_DIMENSIONS)
        return (2 * tp) / (2 * tp + fp + fn + epsilon)
 
    @staticmethod
    def get_dice_loss(output, target, epsilon=1e-9):
        return 1. - UNetValidationPlan.get_dice_score(output, target, epsilon)
 
    # Data loading ----------------------------------------------------------- 
    def training_data(self):
        common_shape = (48, 60, 48)
 
        image_transform = Compose([
            EnsureChannelFirst(channel_dim="no_channel"),
            Resize(common_shape),
            NormalizeIntensity(),
        ])
        target_transform = Compose([
            EnsureChannelFirst(channel_dim="no_channel"),
            Resize(common_shape),
            AsDiscrete(to_onehot=2),
        ])
 
        # shuffle=False: nothing is learned, and a deterministic order makes
        # per-sample results reproducible from one run to another.
        loader_arguments = {'shuffle': False}
 
        mf = MedicalFolderDataset(
            data_modalities='T1',
            target_modalities='label',
            transform={'T1': image_transform},
            target_transform={'label': target_transform},
        )
        return DataManager(mf, **loader_arguments)
 
    # No training only inference -----------------------------------------------
    def training_step(self, data, target):
        pass
        
 
    # Testing calculation ------------------------------------------------------
    def testing_step(self, data, target):
        """
        Called on the nodes for validation. self.eval() is already applied by
        Fed-BioMed. Returns a dict -> the keys become the metric names shown in
        the logs, in the Monitor and in Tensorboard.
        """
        img = data['T1']
        y = target['label']
 
        with torch.no_grad():
            prediction = self.model().forward(img)
            dice = UNetValidationPlan.get_dice_score(prediction, y)  # (B, C)
 
        return {
            'DICE_LOSS':       float((1. - dice).mean().item()),
            'DICE_BACKGROUND': float(dice[:, 0].mean().item()),
            'DICE_BRAIN':      float(dice[:, 1].mean().item()),
        }

In [15]:
# --------------------------------------------------------------------------- #
#  Experiment arguments
# --------------------------------------------------------------------------- #
# Must match the architecture of the weights you are importing.
model_args = {
    'in_channels': 1,
    'out_classes': 2,
    'dimensions': 3,
    'num_encoding_blocks': 3,
    'out_channels_first_layer': 8,
    'normalization': 'batch',
    'upsampling_type': 'linear',
    'padding': True,
    'activation': 'PReLU',
}
 
training_args = {
    'loader_args': {'batch_size': 16},
    'optimizer_args': {'lr': 0.1},      # frozen model
    'epochs': 5,                        # ignored: the train split is empty
    'dry_run': False,
    'log_interval': 1,
    'random_seed': 1234,
 
    # ---- validation section ----
    'test_ratio': 1.0,                  # 100% of local data used for validation
    'test_on_global_updates': False,     # evaluate the model received from the researcher
    'test_on_local_updates': True,     # nothing is learned locally -> disabled
    'test_batch_size': 4,
    'shuffle_testing_dataset': False,
}
 
tags = ['brain-segmentation']       # or 'ixi-holdout' if you deployed the holdout folders
num_rounds = 1            # 1 round -> validation before and after (identical here)
 

# --------------------------------------------------------------------------- #
#  Build, load initial weights, run
# --------------------------------------------------------------------------- #
exp = Experiment(
    tags=tags,
    model_args=model_args,
    training_plan_class=UNetValidationPlan,
    training_args=training_args,
    round_limit=num_rounds,
    aggregator=FedAverage(),
    tensorboard=True,
)

2026-09-07 15:44:28,524 fedbiomed INFO - Updating training data. This action will update FederatedDataset, and the nodes that will participate to the experiment.

2026-09-07 15:44:28,532 fedbiomed INFO - Node selected for training -> Default Node Name
Node ID is -> NODE_b6662ab9-8029-4995-bfa0-af3a5b68a18f

2026-09-07 15:44:28,533 fedbiomed INFO - Node selected for training -> Default Node Name
Node ID is -> NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1

In [2]:
# Load pre-trained model this model will be sent to each node for inference 
exp.training_plan().import_model('brain-segmentation-trained-model')

NameError: name 'exp' is not defined

In [ ]:
# Running an experiment is going to log validation results at each batch of data. 
exp.run()

2026-09-07 15:44:33,040 fedbiomed INFO - Sampled nodes in round 0 ['NODE_b6662ab9-8029-4995-bfa0-af3a5b68a18f', 'NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1']

2026-09-07 15:44:33,045 fedbiomed INFO - Sending request 
					 To: NODE_b6662ab9-8029-4995-bfa0-af3a5b68a18f 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-09-07 15:44:33,046 fedbiomed INFO - Sending request 
					 To: NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-09-07 15:44:36,427 fedbiomed INFO - VALIDATION ON LOCAL UPDATES 
					 NODE_ID: NODE_b6662ab9-8029-4995-bfa0-af3a5b68a18f 
					 Node Name: Default Node Name 
					 Round 1 | Iteration: 1/69 (1%) | Samples: 4/276
 					 DICE_BACKGROUND: 0.691936 
					 DICE_LOSS: 0.582604 
					 DICE_BRAIN: 0.142855 
					 ---------

2026-09-07 15:44:36,445 fedbiomed INFO - VALIDATION ON LOCAL UPDATES 
					 NODE_ID: NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1 
					 Node Name: Default Node Name 
					 Round 1 | Iteration: 1/40 (2%) | Samples: 4/159
 					 DICE_BACKGROUND: 0.690985 
					 DICE_LOSS: 0.579067 
					 DICE_BRAIN: 0.150881 
					 ---------

...

2026-09-07 15:45:08,719 fedbiomed INFO - VALIDATION ON LOCAL UPDATES 
					 NODE_ID: NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1 
					 Node Name: Default Node Name 
					 Round 1 | Iteration: 36/40 (90%) | Samples: 144/159
 					 DICE_BACKGROUND: 0.683986 
					 DICE_LOSS: 0.599294 
					 DICE_BRAIN: 0.117426 
					 ---------

2026-09-07 15:45:09,195 fedbiomed INFO - VALIDATION ON LOCAL UPDATES 
					 NODE_ID: NODE_b6662ab9-8029-4995-bfa0-af3a5b68a18f 
					 Node Name: Default Node Name 
					 Round 1 | Iteration: 37/69 (54%) | Samples: 148/276
 					 DICE_BACKGROUND: 0.689587 
					 DICE_LOSS: 0.577361 
					 DICE_BRAIN: 0.155691 
					 ---------

2026-09-07 15:45:09,687 fedbiomed INFO - VALIDATION ON LOCAL UPDATES 
					 NODE_ID: NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1 
					 Node Name: Default Node Name 
					 Round 1 | Iteration: 37/40 (92%) | Samples: 148/159
 					 DICE_BACKGROUND: 0.684691 
					 DICE_LOSS: 0.597696 
					 DICE_BRAIN: 0.119918 
					 ---------

2026-09-07 15:45:10,127 fedbiomed INFO - VALIDATION ON LOCAL UPDATES 
					 NODE_ID: NODE_b6662ab9-8029-4995-bfa0-af3a5b68a18f 
					 Node Name: Default Node Name 
					 Round 1 | Iteration: 38/69 (55%) | Samples: 152/276
 					 DICE_BACKGROUND: 0.667539 
					 DICE_LOSS: 0.623556 
					 DICE_BRAIN: 0.085349 
					 ---------

2026-09-07 15:45:10,618 fedbiomed INFO - VALIDATION ON LOCAL UPDATES 
					 NODE_ID: NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1 
					 Node Name: Default Node Name 
					 Round 1 | Iteration: 38/40 (95%) | Samples: 152/159
 					 DICE_BACKGROUND: 0.686469 
					 DICE_LOSS: 0.589280 
					 DICE_BRAIN: 0.134971 
					 ---------